# Backpropagation Through Time (BPTT)

Companion notebook for the [BPTT wiki page](https://ml-viz-ruby.vercel.app/wiki/bptt-algorithm).

We build a 1-layer vanilla RNN **from scratch** in NumPy, implement full BPTT manually,
and compare our hand-rolled gradients against PyTorch autograd.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Dark matplotlib style
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#888',
    'ytick.color':      '#888',
    'text.color':       '#eee',
    'grid.color':       '#333',
    'lines.linewidth':  2,
})

np.random.seed(42)

## 1 — RNN forward pass from scratch

The recurrence is:
$$h_t = \tanh(W_{hh}h_{t-1} + W_{xh}x_t + b)$$

We store the pre-activations $a_t$ and hidden states $h_t$ needed for BPTT.

In [ ]:
def rnn_forward(x_seq, W_hh, W_xh, b, h0=None):
    """Forward pass through a 1-layer scalar RNN.
    
    x_seq: (T,) input sequence
    Returns: h (T+1,), a (T,)
    """
    T = len(x_seq)
    h = np.zeros(T + 1)
    a = np.zeros(T)
    h[0] = h0 if h0 is not None else 0.0
    for t in range(T):
        a[t] = W_hh * h[t] + W_xh * x_seq[t] + b
        h[t+1] = np.tanh(a[t])
    return h, a

W_hh, W_xh, b = 0.8, 0.5, 0.0
x_seq = np.array([1.0, 0.0, 1.0])

h, a = rnn_forward(x_seq, W_hh, W_xh, b)
tanh_prime = 1 - np.tanh(a)**2

print("Hidden states h_0..h_3:", h.round(3))
print("Pre-activations a_1..a_3:", a.round(3))
print("tanh' values:", tanh_prime.round(3))

## 2 — Manual BPTT

Given a loss $\mathcal{L} = \frac{1}{T}\sum_t h_t^2$ (a toy regression target of zero),
we manually compute $\partial\mathcal{L}/\partial W_{hh}$ by accumulating contributions
from all time steps.

In [ ]:
def bptt(x_seq, h, a, W_hh, W_xh):
    """Full BPTT for loss = mean(h^2) over all steps."""
    T = len(x_seq)
    tanh_p = 1 - np.tanh(a)**2

    dW_hh = 0.0
    dh = np.zeros(T + 1)

    # Upstream gradient from loss: dL/dh_t = 2*h_t / T for t=1..T
    for t in range(1, T + 1):
        dh[t] = 2 * h[t] / T

    # Backprop through time (reverse)
    for t in reversed(range(T)):
        da_t = dh[t+1] * tanh_p[t]          # dL/da_t
        dW_hh += da_t * h[t]                 # dL/dW_hh accumulates
        dh[t] += da_t * W_hh                 # pass gradient back to h_{t-1}

    return dW_hh

dW_hh_manual = bptt(x_seq, h, a, W_hh, W_xh)
print(f"Manual BPTT dW_hh = {dW_hh_manual:.6f}")

## 3 — Verify with finite differences

In [ ]:
def loss_fn(W_hh_val):
    h_, _ = rnn_forward(x_seq, W_hh_val, W_xh, b)
    return np.mean(h_[1:]**2)

eps = 1e-5
dW_hh_fd = (loss_fn(W_hh + eps) - loss_fn(W_hh - eps)) / (2 * eps)

print(f"Finite-difference dW_hh = {dW_hh_fd:.6f}")
print(f"Relative error = {abs(dW_hh_manual - dW_hh_fd) / (abs(dW_hh_fd) + 1e-12):.2e}")

## 4 — Gradient norm vs. sequence length

We plot how the upstream gradient at step 1 decays as the sequence gets longer,
for several values of $W_{hh}$.

In [ ]:
def grad_at_step1(W_hh_val, T):
    """Gradient flowing back to h_1 from a unit loss at h_T."""
    x = np.zeros(T)
    x[0] = 1.0
    h_, a_ = rnn_forward(x, W_hh_val, 0.5, 0.0)
    tanh_p = 1 - np.tanh(a_)**2
    # Product of per-step Jacobians (scalar case)
    grad = 1.0
    for t in reversed(range(1, T)):
        grad *= tanh_p[t] * W_hh_val
    return abs(grad)

T_range = range(1, 31)
configs = [(0.5, '#6366f1', 'λ=0.5 (vanishing)'),
           (1.0, '#20d9d2', 'λ=1.0 (neutral)'),
           (1.3, '#f97316', 'λ=1.3 (exploding)')]

fig, ax = plt.subplots(figsize=(9, 5))
for W_val, color, label in configs:
    grads = [grad_at_step1(W_val, T) for T in T_range]
    ax.semilogy(list(T_range), grads, color=color, label=label)

ax.set_xlabel('Sequence length T')
ax.set_ylabel('|grad at step 1| (log scale)')
ax.set_title('BPTT gradient magnitude vs. sequence length')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## ✏️ Your turn

### Exercise 1 — Multi-step gradient product

Compute $\partial h_T / \partial h_1$ analytically (as a product of per-step Jacobians)
and verify against finite differences for $T=5$, $W_{hh}=0.9$.

In [ ]:
# TODO(you): compute the product of per-step Jacobians for T=5, W_hh=0.9
T, W_val = 5, 0.9
x_test = np.ones(T)

# Forward pass to get tanh' values
h_test, a_test = rnn_forward(x_test, W_val, 0.5, 0.0)
tanh_p_test = 1 - np.tanh(a_test)**2

# --- Analytic: product of Jacobians from step T down to step 2 ---
analytic = None  # TODO: compute as product over t=1..T-1 of tanh_p[t]*W_val

# --- Finite difference: perturb h_1 and measure change in h_T ---
eps = 1e-5
def h_T_from_h1(h1_val):
    h_fd = np.zeros(T + 1)
    h_fd[1] = h1_val
    for t in range(1, T):
        a_t = W_val * h_fd[t] + 0.5 * x_test[t]
        h_fd[t+1] = np.tanh(a_t)
    return h_fd[T]

finite_diff = (h_T_from_h1(h_test[1] + eps) - h_T_from_h1(h_test[1] - eps)) / (2 * eps)

# assert abs(analytic - finite_diff) / (abs(finite_diff) + 1e-12) < 1e-4, "Mismatch!"
# print(f"analytic={analytic:.5f}, finite_diff={finite_diff:.5f} ✓")

### Exercise 2 — Truncated BPTT

Implement truncated BPTT with window size $k=3$: detach the hidden state every $k$ steps
and compare the accumulated $\partial\mathcal{L}/\partial W_{hh}$ to full BPTT for $T=12$.

<details>
<summary>Solution hint</summary>

```python
# Full BPTT accumulates gradients over the whole sequence.
# Truncated BPTT breaks the sequence into chunks of length k,
# resets the gradient to zero at each chunk boundary,
# and detaches (stops gradient through) the hidden state at boundaries.
# The weight gradient from each chunk is smaller but still points in
# approximately the right direction for nearby dependencies.
```
</details>